In [1]:
import os
import re
import pandas as pd
import numpy as np

# 1、原始数据:reads_1.fq 和 reads_2.fq

In [ ]:
name_dict = {
    1: '3dpi_N289D_C57mouse_Lung',
    2: '3dpi_T7_C57mouse_Lung',
    3: '3dpi_HY_C57mouse_Lung',
    4: '3dpi_0_C57mouse_Lung',
    5: '3dpi_KT_C57mouse_Lung',
    6: '5dpi_N289D_C57mouse_Lung',
    7: '5dpi_T7_C57mouse_Lung',
    8: '5dpi_0_C57mouse_Lung',
    9: '5dpi_HY_C57mouse_Lung',
    10: '5dpi_KT_C57mouse_Lung',
}

src_path = '/data/shenlab_data/01.Data_Sequencing/030.HA转录组20240123/DATA_20240121023900_1/data'

for i in range(1, 11):
    new_name = name_dict[i]
    i = f'0{i}' if i < 10 else f'{i}'
    for j in range(1, 3):
        print(j)
        path_0 = f'{src_path}/Unknown_BU999-001T00{i}_good_{j}.fq.gz'
        path_1 = f'data/{new_name}_{j}.fq.gz'
        cmd = f'ln -s {path_0} {path_1}'
        
        if os.path.exists(path_0):
            os.system(cmd)

# 2、使用Hisat2将reads映射到参考基因组

输入文件:reads_1.fq, reads_2.fq, 参考基因组基因组索引文件

输出文件:alignments.sam

hisat2 -p 8 -x genome_index -1 reads_1.fq -2 reads_2.fq -S alignments.samhisat2 -p 8 -x genome_index -1 reads_1.fq -2 reads_2.fq -S alignments.sam

In [369]:
grcm38_index = 'assets/grcm38_tran/genome_tran'

cmd_list = []

for file in os.listdir('data'):
    
    if not file.endswith('_1.fq.gz'):
        continue
        
    sample = file.replace('_1.fq.gz', '')
    
    hisat2_cmd = f'hisat2 --dta -p 4 -x {grcm38_index} -1 data/{sample}_1.fq.gz -2 data/{sample}_2.fq.gz -S outputs/hisat2_outputs/{sample}.hisat2.sam'
    
    cmd_list.append(hisat2_cmd)
    
print(' & '.join(cmd_list))

hisat2 --dta -p 4 -x assets/grcm38_tran/genome_tran -1 data/5dpi_KT_C57mouse_Lung_1.fq.gz -2 data/5dpi_KT_C57mouse_Lung_2.fq.gz -S outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.hisat2.sam & hisat2 --dta -p 4 -x assets/grcm38_tran/genome_tran -1 data/3dpi_N289D_C57mouse_Lung_1.fq.gz -2 data/3dpi_N289D_C57mouse_Lung_2.fq.gz -S outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.sam & hisat2 --dta -p 4 -x assets/grcm38_tran/genome_tran -1 data/3dpi_0_C57mouse_Lung_1.fq.gz -2 data/3dpi_0_C57mouse_Lung_2.fq.gz -S outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.sam & hisat2 --dta -p 4 -x assets/grcm38_tran/genome_tran -1 data/5dpi_N289D_C57mouse_Lung_1.fq.gz -2 data/5dpi_N289D_C57mouse_Lung_2.fq.gz -S outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.hisat2.sam & hisat2 --dta -p 4 -x assets/grcm38_tran/genome_tran -1 data/5dpi_HY_C57mouse_Lung_1.fq.gz -2 data/5dpi_HY_C57mouse_Lung_2.fq.gz -S outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.sam & hisat2 --dta -p 4 -x assets/grcm38_tran/

# 3、将sam文件转换为bam文件

输入文件:alignments.sam

输出文件:alignments.bam

samtools view -Sb alignments.sam > alignments.bamsamtools view -Sb alignments.sam > alignments.bam

In [377]:
for sam_file in os.listdir('outputs/hisat2_outputs'):
    
    if not sam_file.endswith('.hisat2.sam'):
        continue
        
    cmd_0 = f'samtools view -bS outputs/hisat2_outputs/{sam_file} > outputs/hisat2_outputs/{sam_file.replace(".sam", "")}.bam'
        
    print(cmd_0)
    
    os.system(cmd_0)
    

samtools view -bS outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.hisat2.bam
samtools view -bS outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.hisat2.s

In [ ]:
cmds = []

for sam_file in os.listdir('outputs/hisat2_outputs'):
    
    if not sam_file.endswith('.hisat2.sam'):
        continue
        
    sample = sam_file.replace('.hisat2.sam', '')
    
    cmd_0 = f'samtools view -bS outputs/hisat2_outputs/{sam_file} > outputs/hisat2_outputs/{sample}.bam' # 将SAM文件转换为BAM文件
    cmd_1 = f'samtools sort outputs/hisat2_outputs/{sample}.bam -o outputs/hisat2_outputs/{sample}.sorted.bam' # 对BAM文件进行排序
    cmd_2 = f'samtools index outputs/hisat2_outputs/{sample}.sorted.bam' # 为BAM文件创建索引
    
    cmd = f'{cmd_0} && {cmd_1} && {cmd_2}'
    
    print(f'{cmd}\n')
    
    os.system(cmd)

samtools view -bS outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.bam



Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.
Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.


samtools view -bS outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.bam

samtools view -bS outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.bam



Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.
Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.


samtools view -bS outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.bam

samtools view -bS outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.bam



Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.
Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.


samtools view -bS outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.bam

samtools view -bS outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.bam



Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.
Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.


samtools view -bS outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.bam

samtools view -bS outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.bam

samtools view -bS outputs/hisat2_outputs/5dpi_T7_C57mouse_Lung.hisat2.sam > outputs/hisat2_outputs/5dpi_T7_C57mouse_Lung.bam



Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.
Error, dont recognize --est_method featureCounts  at /home/chenrujian/trinityrnaseq-v2.15.1/util/abundance_estimates_to_matrix.pl line 81.


# 4、使用featureCounts计算基因表达量

输入文件:alignments.bam, 基因注释文件gene.gtf

输出文件:counts.txt

featureCounts -a gene.gtf -o counts.txt alignments.bamfeatureCounts -a gene.gtf -o counts.txt alignments.bam

In [ ]:
annotation_gtf = 'assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf'

counts_txt = f'outputs/featureCounts_outputs/featurecounts.txt'

featureCounts_cmd = f'featureCounts -a {annotation_gtf} -o {counts_txt} -p -t exon -g gene_id'

for sam_file in os.listdir('outputs/hisat2_outputs'):
    
    if not sam_file.endswith('.sorted.bam'):
        continue
        
    sample = sam_file.replace('.sorted.bam', '')
    sample_bam = f'outputs/hisat2_outputs/{sample}.sorted.bam'
    
    featureCounts_cmd += f' {sample_bam}'
    
print(featureCounts_cmd)
    
os.system(featureCounts_cmd)

featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/5dpi_T7_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.sorted.bam outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.sorted.bam



        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 10 BAM files                                     ||
||                                                                            ||
||                           5dpi_N289D_C57mouse_Lung.sorted.bam              ||
||                           3dpi_KT_C57mouse_Lung.sorted.bam                 ||
||                           5dpi_HY_C57mouse_Lung.sorted.bam    

0

In [ ]:
annotation_gtf = 'assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf'

for sam_file in os.listdir('outputs/hisat2_outputs'):
    
    if not sam_file.endswith('.sorted.bam'):
        continue
        
    sample = sam_file.replace('.sorted.bam', '')
    sample_bam = f'outputs/hisat2_outputs/{sample}.sorted.bam'
    counts_txt = f'outputs/featureCounts_outputs/{sample}.featurecounts.txt'
    
    featureCounts_cmd = f'featureCounts -a {annotation_gtf} -o {counts_txt} -p -t exon -g gene_id {sample_bam}'
    
    print(featureCounts_cmd, '\n')
    
    os.system(featureCounts_cmd)


featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/5dpi_N289D_C57mouse_Lung.featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.sorted.bam 




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           5dpi_N289D_C57mouse_Lung.sorted.bam              ||
||                                                                            ||
||             Output file : 5dpi_N289D_C57mouse_Lung.featurecoun

featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/3dpi_KT_C57mouse_Lung.featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.sorted.bam 



||    Features : 710016                                                       ||
||    Meta-features : 47729                                                   ||
||    Chromosomes/contigs : 45                                                ||
||                                                                            ||
|| Process BAM file 3dpi_KT_C57mouse_Lung.sorted.bam...                       ||
||    Paired-end reads are included.                                          ||
||    The reads are assigned on the single-end mode.                          ||
||    Total alignments : 64475396                                             ||
||    Successfully assigned alignments : 51178912 (79.4%)                     ||
||    Running time : 9.74 minutes                                             ||
||                                                                            ||
|| Write the final count table.                                               ||
|| Write the read assignment

featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/5dpi_HY_C57mouse_Lung.featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.sorted.bam 



...

||    Features : 710016                                                       ||
||    Meta-features : 47729                                                   ||
||    Chromosomes/contigs : 45                                                ||
||                                                                            ||
|| Process BAM file 3dpi_0_C57mouse_Lung.sorted.bam...                        ||
||    Paired-end reads are included.                                          ||
||    The reads are assigned on the single-end mode.                          ||
||    Total alignments : 66806874                                             ||
||    Successfully assigned alignments : 45494635 (68.1%)                     ||
||    Running time : 9.07 minutes                                             ||
||                                                                            ||
|| Write the final count table.                                               ||
|| Write the read assignment

featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/3dpi_N289D_C57mouse_Lung.featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.sorted.bam 




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           3dpi_N289D_C57mouse_Lung.sorted.bam              ||
||                                                                            ||
||             Output file : 3dpi_N289D_C57mouse_Lung.featurecoun

featureCounts -a assets/grcm38_tran/Mus_musculus.GRCm38.84.gtf -o outputs/featureCounts_outputs/5dpi_KT_C57mouse_Lung.featurecounts.txt -p -t exon -g gene_id outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.sorted.bam 




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
	  v2.0.6

//========================== featureCounts setting ===========================\\
||                                                                            ||
||             Input files : 1 BAM file                                       ||
||                                                                            ||
||                           5dpi_KT_C57mouse_Lung.sorted.bam                 ||
||                                                                            ||
||             Output file : 5dpi_KT_C57mouse_Lung.featurecounts.

In [345]:
rsem_script_path = '/home/chenrujian/RSEM-1.3.3'

In [394]:
# 构建索引
genome_fa = 'assets/mus_genome/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.fna'
annotation_gtf = 'assets/mus_annotations/GCF_000001635.27/genomic.gtf'
reference_name = 'GRCm39_genomic_reference'

rsem_build = f'{rsem_script_path}/rsem-prepare-reference -p 40 \
--gtf {annotation_gtf} \
{genome_fa} {reference_name}'

print(rsem_build)
os.system(rsem_build)

/home/chenrujian/RSEM-1.3.3/rsem-prepare-reference -p 40 --gtf assets/mus_annotations/GCF_000001635.27/genomic.gtf assets/mus_genome/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.fna GRCm39_genomic_reference
rsem-extract-reference-transcripts GRCm39_genomic_reference 0 assets/mus_annotations/GCF_000001635.27/genomic.gtf None 0 assets/mus_genome/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.fna
Parsed 200000 lines
Parsed 400000 lines
Parsed 600000 lines
Parsed 800000 lines
Parsed 1000000 lines
Parsed 1200000 lines
Parsed 1400000 lines
Parsed 1600000 lines
Parsed 1800000 lines
Parsed 2000000 lines
Parsed 2200000 lines
Parsed 2400000 lines
Parsed 2600000 lines
Parsed 2800000 lines
Parsed 3000000 lines
Parsing gtf File is done!
assets/mus_genome/data/GCF_000001635.27/GCF_000001635.27_GRCm39_genomic.fna is processed!
137925 transcripts are extracted.
Extracting sequences is done!
Group File is generated!
Transcript Information File is generated!
Chromosome List File is gener

0

In [380]:
upstream_read_bams = []

for sample in os.listdir('outputs/hisat2_outputs'):
    if not sample.endswith(".bam"):
        continue
    p = f'outputs/hisat2_outputs/{sample}'
    upstream_read_bams.append(p)
    
upstream_read_bams

['outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_T7_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.bam']

In [382]:
upstream_read_bams = ['outputs/hisat2_outputs/3dpi_HY_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_T7_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_KT_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_T7_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_KT_C57mouse_Lung.hisat2.bam',
 #'outputs/hisat2_outputs/5dpi_0_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_N289D_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_N289D_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.hisat2.bam',
 'outputs/hisat2_outputs/5dpi_HY_C57mouse_Lung.hisat2.bam']
with open('bam_paths.txt', 'w') as f:
    for bam in upstream_read_bams:
        f.write(f'{bam}\n')

In [ ]:
upstream_read_bams = 'bam_paths.txt'

reference_name = 'GRCm39_genomic_reference'

out_prefix = 'outputs/rsem'

rsem_calculate = f'{rsem_script_path}/rsem-calculate-expression -p 40 \
--hisat2-hca \
--hisat2-path /home/chenrujian/anaconda3/bin/hisat2 \
--paired-end -no-bam-output --alignments \
--bam {upstream_read_bams} {reference_name} {out_prefix}'

print(rsem_calculate)
os.system(rsem_calculate)

/home/chenrujian/RSEM-1.3.3/rsem-calculate-expression -p 40 --hisat2-hca --hisat2-path /home/chenrujian/anaconda3/bin/hisat2 --paired-end -no-bam-output --alignments --bam bam_paths.txt GRCm39_genomic_reference outputs/rsem
rsem-parse-alignments GRCm39_genomic_reference outputs/rsem.temp/rsem outputs/rsem.stat/rsem bam_paths.txt 3 -tag XM
"rsem-parse-alignments GRCm39_genomic_reference outputs/rsem.temp/rsem outputs/rsem.stat/rsem bam_paths.txt 3 -tag XM" failed! Plase check if you provide correct parameters/options for the pipeline!


The SAM/BAM file declares less than one reference sequence!


65280

In [350]:
!/home/chenrujian/RSEM-1.3.3/rsem-calculate-expression --help

NAME
    rsem-calculate-expression - Estimate gene and isoform expression from
    RNA-Seq data.

SYNOPSIS
     rsem-calculate-expression [options] upstream_read_file(s) reference_name sample_name 
     rsem-calculate-expression [options] --paired-end upstream_read_file(s) downstream_read_file(s) reference_name sample_name 
     rsem-calculate-expression [options] --alignments [--paired-end] input reference_name sample_name

ARGUMENTS
    upstream_read_files(s)
        Comma-separated list of files containing single-end reads or
        upstream reads for paired-end data. By default, these files are
        assumed to be in FASTQ format. If the --no-qualities option is
        specified, then FASTA format is expected.

    downstream_read_file(s)
        Comma-separated list of files containing downstream reads which are
        paired with the upstream reads. By default, these files are assumed
        to be in FASTQ format. If the --no-qualities option is specified,
        then FAST

# 5、获得表达量矩阵：genes.TMM.EXPR.matrix

abundance_estimates_to_matrix.pl  \
    --est_method featureCounts \
    --gene_trans_map gene_trans_map.txt \
    --quant_files genes.quant_files.txt \
    --out_prefix genes

In [274]:
with open(annotation_gtf, 'r') as f:
    lines = f.readlines()

In [ ]:
# remove featureCounts_outputs header
for file in os.listdir('outputs/featureCounts_outputs/'):
    if not file.endswith('.featurecounts.txt'):
        continue

    with open(f'outputs/featureCounts_outputs/{file}', 'r') as f:
        a =f.readlines()

    with open(f'outputs/featureCounts_outputs/{file.replace(".txt", ".noheader.txt")}', 'w') as f:
        for line in a[1:]:
            f.write(line)

In [325]:
with open('grcm38_gene_trans_map.txt', 'w') as f:
    for line in lines:
        line_info = line.split('\t')
        if len(line_info) == 1:
            continue
            
        if line_info[2] != 'transcript':
            continue
            
        gene_id = re.search('gene_id "(.*?)"', lines[i]).group(1)
        transcript_id = re.search('transcript_id "(.*?)"', lines[i]).group(1)

        f.write(f'{gene_id}\t{transcript_id}\n')

In [ ]:
def cal_fpkm(featureCounts_txt):
    
    # featureCounts
    
    featureCounts_df = pd.read_csv(featureCounts_txt, sep = '\t')

    featureCounts_df['FPKM'] = 0
    
    count_column = featureCounts_df.columns[-2]
    
    read_sum = featureCounts_df[count_column].sum()
    
    for i in featureCounts_df.index:

        read_count = featureCounts_df.loc[i, count_column]

        gene_length = featureCounts_df.loc[i, 'Length']

        fpkm = 10**9 * read_count / (read_sum * gene_length)
        
        featureCounts_df.loc[i, 'FPKM'] = fpkm

    return featureCounts_df

fpkm_matrix_df = pd.DataFrame()

for sample in ['3dpi_0', '5dpi_0',
               '3dpi_HY', '5dpi_HY', 
               '3dpi_T7', '5dpi_T7', 
               '3dpi_KT', '5dpi_KT',
               '3dpi_N289D', '5dpi_N289D']:
    
    print(sample)
    
    df = cal_fpkm(f'outputs/featureCounts_outputs/{sample}_C57mouse_Lung.featurecounts.noheader.txt')
    
    for i in df.index:
        gene_id = df.loc[i, 'Geneid']
        fpkm = df.loc[i, 'FPKM']
        
        fpkm_matrix_df.loc[gene_id, sample] = fpkm

3dpi_0
5dpi_0
3dpi_HY
5dpi_HY
3dpi_T7
5dpi_T7
3dpi_KT
5dpi_KT
3dpi_N289D
5dpi_N289D


In [ ]:
np.log2(fpkm_matrix_df+1).to_csv('outputs/featureCounts_outputs/fpkm.log2.matrix', sep = '\t')

In [4]:
fpkm_matrix_log2_df = pd.read_csv('outputs/featureCounts_outputs/fpkm.log2.matrix', sep = '\t', index_col = 0)
fpkm_matrix_log2_df

,3dpi_0,5dpi_0,3dpi_HY,5dpi_HY,3dpi_T7,5dpi_T7,3dpi_KT,5dpi_KT,3dpi_N289D,5dpi_N289D
ENSMUSG00000102693,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000064842,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000051951,0.010370,0.129302,0.045973,0.099475,0.013295,0.064096,0.102652,0.050408,0.020927,0.027729
ENSMUSG00000102851,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000103377,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000094431,0.000000,0.000000,0.000000,0.000000,0.000000,0.323866,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000094621,1.493944,0.772601,0.302128,0.227983,0.302338,1.717209,0.403773,0.000000,0.000000,0.665765
ENSMUSG00000098647,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000096730,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


# 6、差异表达分析

输入文件:counts.matrix

输出文件:diff_expressed_genes.txt

run_DE_analysis.pl --matrix counts.matrix --method edgeR --dispersion 0.1run_DE_analysis.pl --matrix counts.matrix --method edgeR --dispersion 0.1

In [3]:
# run_DE_analysis.pl

DE_script = 'trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/run_DE_analysis.pl'

#cmd = f'perl {DE_script} --matrix {counts_txt} --method DESeq2 --samples_file {sample_txt} --contrasts {contrasts_txt} --output {DESeq2_results}'

for vs in ['0', 'HY']:

    for day in ['3', '5']:
        
        sample_txt = f'sample_vs{vs}_{day}dpi.rename.txt'
        
        contrasts_txt = f'contrasts_vs{vs}_{day}dpi.txt'
        
        method = 'edgeR'
        
        counts_txt = 'featurecounts.txt'
        
        cmd = f'perl {DE_script} --matrix {counts_txt} --method {method} --dispersion 0.1 --samples_file {sample_txt} --contrasts {contrasts_txt} --output vs_{vs}/DEG_{method}_results_{day}dpi'
        
        print(cmd)
        
        os.system(cmd)

perl trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/run_DE_analysis.pl --matrix featurecounts.txt --method edgeR --dispersion 0.1 --samples_file sample_vs0_3dpi.rename.txt --contrasts contrasts_vs0_3dpi.txt --output vs_0/DEG_edgeR_results_3dpi
$VAR1 = {
          '3dpi_HY' => [
                         '3dpi_HY'
                       ],
          '3dpi_T7' => [
                         '3dpi_T7'
                       ],
          '3dpi_N289D' => [
                            '3dpi_N289D'
                          ],
          '3dpi_0' => [
                        '3dpi_0'
                      ],
          '3dpi_KT' => [
                         '3dpi_KT'
                       ]
        };
CMD: Rscript featurecounts.txt.3dpi_HY_vs_3dpi_0.3dpi_HY.vs.3dpi_0.EdgeR.Rscript


Got 16 samples, and got: 16 data fields.
Header: Geneid	Chr	Start	End	Strand	Length	5dpi_N289D	3dpi_KT	5dpi_HY	3dpi_HY	5dpi_T7	5dpi_0	3dpi_T7	3dpi_0	3dpi_N289D	5dpi_KT
Next: ENSMUSG00000102693	1	3073253	3074322	+	1070	0	0	0	0	0	0	0	0	0	0

-shifting sample indices over.
$VAR1 = {
          'Strand' => 4,
          '5dpi_HY' => 8,
          '3dpi_KT' => 7,
          '3dpi_T7' => 12,
          '5dpi_N289D' => 6,
          '3dpi_0' => 13,
          'End' => 3,
          '5dpi_0' => 11,
          'Length' => 5,
          '3dpi_N289D' => 14,
          '5dpi_T7' => 10,
          '3dpi_HY' => 9,
          '5dpi_KT' => 15,
          'Start' => 2,
          'Chr' => 1
        };
Contrasts to perform are: $VAR1 = [
          [
            '3dpi_HY',
            '3dpi_0'
          ],
          [
            '3dpi_KT',
            '3dpi_0'
          ],
          [
            '3dpi_T7',
            '3dpi_0'
          ],
          [
            '3dpi_N289D',
            '3dpi_0'
          ]
        

null device 
          1 
CMD: Rscript featurecounts.txt.3dpi_KT_vs_3dpi_0.3dpi_KT.vs.3dpi_0.EdgeR.Rscript


Loading required package: edgeR
Loading required package: limma


null device 
          1 
CMD: Rscript featurecounts.txt.3dpi_T7_vs_3dpi_0.3dpi_T7.vs.3dpi_0.EdgeR.Rscript


...

null device 
          1 
CMD: Rscript featurecounts.txt.5dpi_T7_vs_5dpi_HY.5dpi_T7.vs.5dpi_HY.EdgeR.Rscript


Loading required package: edgeR
Loading required package: limma


null device 
          1 
CMD: Rscript featurecounts.txt.5dpi_N289D_vs_5dpi_HY.5dpi_N289D.vs.5dpi_HY.EdgeR.Rscript


Loading required package: edgeR
Loading required package: limma


null device 
          1 


In [18]:
pd.read_csv('FPKM.csv', index_col=0)

,3dpi_0,3dpi_HY,3dpi_N289D,3dpi_K412R/T480A,3dpi_ΔL226/R229I,5dpi_0,5dpi_HY,5dpi_N289D,5dpi_K412R/T480A,5dpi_ΔL226/R229I
ENSMUSG00000051951,0.165957,1.353919,0.004391,0.107838,0.012906,0.060900,0.046425,0.053060,0.186240,0.228792
ENSMUSG00000102331,0.004346,0.264897,0.000000,0.000000,0.000000,0.033157,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000102343,0.006453,0.000000,0.000000,0.000000,0.000000,0.098722,0.000000,0.000000,0.004185,0.000000
ENSMUSG00000025900,1.451405,30.608244,1.003202,0.699032,14.612404,6.153525,4.823649,9.131228,2.178156,1.762423
ENSMUSG00000025902,5.711609,167.324596,164.156387,8.259482,35.051636,36.521312,3.456206,5.071236,11.399467,10.857966
...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000063897,12.871712,2.347247,1.558404,0.432667,1.374958,4.720083,60.473749,5.866276,8.584109,2.019462
ENSMUSG00000084520,0.000000,0.006356,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002311,0.000000
ENSMUSG00000094431,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.014568
ENSMUSG00000094621,0.016721,0.026149,0.000000,0.067702,0.005893,0.044209,0.089366,0.250476,0.000000,0.071662


In [19]:
np.log2(pd.read_csv('FPKM.csv', index_col=0)+1).to_csv('FPKM_log2.matrix')

In [ ]:
# analyze_diff_expr.pl

# run in DEG_edgeR_results

for vs in ['0', 'HY']:

    for day in ['3', '5']:

        DE_script = '/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl'

        matrix = '/Users/chenrujian/Desktop/转录组/fpkm.log2.matrix'

        sample_txt = f'/Users/chenrujian/Desktop/转录组/sample_vs{vs}_{day}dpi.rename.txt'

        cmd = f'{DE_script} --matrix {matrix} --samples {sample_txt} -P 0.05 -C 1'

        print(f'{cmd}\n')

/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl --matrix /Users/chenrujian/Desktop/转录组/fpkm.log2.matrix --samples /Users/chenrujian/Desktop/转录组/sample_vs0_3dpi.rename.txt -P 0.05 -C 1

/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl --matrix /Users/chenrujian/Desktop/转录组/fpkm.log2.matrix --samples /Users/chenrujian/Desktop/转录组/sample_vs0_5dpi.rename.txt -P 0.05 -C 1

/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl --matrix /Users/chenrujian/Desktop/转录组/fpkm.log2.matrix --samples /Users/chenrujian/Desktop/转录组/sample_vsHY_3dpi.rename.txt -P 0.05 -C 1

/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl --matrix /Users/chenrujian/Desktop/转录组/fpkm.log2.matrix --samples /Users/chenrujian/Desktop/转录组/sample_vsHY_5dpi.rename.txt -P 0.05 -C 1



In [ ]:
/Users/chenrujian/Desktop/转录组/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/analyze_diff_expr.pl --matrix /Users/chenrujian/Desktop/转录组/featurecounts.txt --samples /Users/chenrujian/Desktop/转录组/sample_vs0_3dpi.rename.txt -P 0.05 -C 1

# 将基因划分为表达簇

  $TRINITY_HOME/Analysis/DifferentialExpression/define_clusters_by_cutting_tree.pl \
                                    -R  diffExpr.P0.001_C2.matrix.RData --Ptree 60

In [884]:
# 在DEG_edgeR_results/（或拆分后的DEG_edgeR_results_split/sample/）目录下执行

cluster_script = '/home/chenrujian/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/define_clusters_by_cutting_tree.pl'

cmd = f'{cluster_script} -R diffExpr.P0.05_C2.matrix.RData --Ptree 60'

print(cmd)

/home/chenrujian/trinityrnaseq-v2.15.1/Analysis/DifferentialExpression/define_clusters_by_cutting_tree.pl -R diffExpr.P0.05_C2.matrix.RData --Ptree 60


In [ ]:
cluster_path = 'outputs/DEG_edgeR_results/diffExpr.P0.001_C2.matrix.RData.clusters_fixed_P_60'

os.system(f'chmod 777 {cluster_path}/__tmp_plot_clusters.R')

for i in os.listdir(cluster_path):
    if not i.endswith('.matrix'):
        continue
    matrix_df = pd.read_csv(os.path.join(cluster_path, i), sep = '\t')
    matrix_df = matrix_df[['3dpi_0', '5dpi_0', 
               '3dpi_HY', '5dpi_HY', 
               '3dpi_KT', '5dpi_KT', 
               '3dpi_T7', '5dpi_T7', 
               '3dpi_N289D', '5dpi_N289D', ]]
    matrix_df.to_csv(os.path.join(cluster_path, i), sep = '\t')

print('Rscript __tmp_plot_clusters.R')

Rscript __tmp_plot_clusters.R


# 功能富集

In [504]:
import pyclusterprofiler
from Ensembl_converter import EnsemblConverter

# Create an instance of EnsemblConverter
converter = EnsemblConverter()

In [ ]:
# cluster
cluster_path = 'outputs/DEG_edgeR_results/diffExpr.P0.001_C2.matrix.RData.clusters_fixed_P_60'
cluster_df = pd.DataFrame()
for file in os.listdir(cluster_path):
    if not file.endswith('.matrix'):
        continue
    sub_df = pd.read_csv(os.path.join(cluster_path, file), sep = '\t', index_col = 0)
    sub_df['cluster'] = file.split('_')[1]
    cluster_df = pd.concat([cluster_df, sub_df], axis = 0)
cluster_df

,3dpi_0,5dpi_0,3dpi_HY,5dpi_HY,3dpi_KT,5dpi_KT,3dpi_T7,5dpi_T7,3dpi_N289D,5dpi_N289D,cluster
ENSMUSG00000066108,0.438599,0.369218,0.497305,-0.926419,0.512990,-0.111107,-0.293476,-1.113640,-0.035127,0.661657,6
ENSMUSG00000029273,1.007265,0.753711,-0.026233,-0.509139,0.467544,-0.102494,-0.960756,-0.334545,-0.557997,0.262642,6
ENSMUSG00000020581,0.241598,-0.070169,0.508851,-0.613420,0.411096,0.167218,-0.473578,-0.471598,-0.401148,0.701151,6
ENSMUSG00000030017,0.477450,0.336721,0.626127,-0.730594,0.597377,-0.120561,-0.597278,-0.720608,-0.577350,0.708715,6
ENSMUSG00000064057,0.762159,0.603964,0.282125,-0.512069,0.277633,0.000931,-0.196490,-1.196406,-0.407299,0.385452,6
...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000030546,1.096168,0.460191,-0.373617,0.646032,0.367679,-0.844594,0.060664,-0.415580,-0.843111,-0.153832,4
ENSMUSG00000027513,1.213209,0.174066,-0.895257,0.699447,0.736912,-1.092472,-0.144049,0.231369,-0.685841,-0.237384,4
ENSMUSG00000095633,-0.647395,0.027543,1.722594,-0.616296,0.328250,0.540762,-0.682634,0.185636,-0.532743,-0.325718,7
ENSMUSG00000076569,0.652913,-0.559763,-0.328888,0.995197,-0.328239,0.443036,-0.871674,0.018343,-0.280933,0.260008,3


In [ ]:
symbol_df = converter.convert_ids(cluster_df.index.tolist())
symbol_df

,ENSG,Symbol
0,ENSMUSG00000066108,Muc5b
1,ENSMUSG00000029273,Sult1d1
2,ENSMUSG00000020581,Agr2
3,ENSMUSG00000030017,Reg3g
4,ENSMUSG00000064057,Scgb3a1
...,...,...
682,ENSMUSG00000030546,Plin1
683,ENSMUSG00000027513,Pck1
684,ENSMUSG00000095633,Igkv4-58
685,ENSMUSG00000076569,Igkv5-39


In [ ]:
id2genename = {}
gene_info_df = pd.read_csv('id2genename.txt', sep = '\t')
gene_info_df['symbol'] = ''
for i in gene_info_df.index:
    gene_name = gene_info_df.loc[i, 'Gene Name']
    symbol = gene_name.split('(')[1].split(')')[0]
    gene_info_df.loc[i, 'symbol'] = symbol
    id2genename[gene_info_df.loc[i, 'From']] = symbol
gene_info_df

In [ ]:
unname_geneid_list = []
for geneid in fpkm_matrix_log2_df.index.tolist():
    if geneid not in gene_info_df['From'].tolist():
        unname_geneid_list.append(geneid)
new_symbol_df = converter.convert_ids(unname_geneid_list)
new_symbol_df

In [ ]:
# symbol
for i in cluster_df.index:
    symbol = symbol_df.loc[symbol_df['ENSG']==i, 'Symbol'].item()
    if symbol == '-':
        cluster_df.loc[i, 'symbol'] = '-'
    else:
        cluster_df.loc[i, 'symbol'] = symbol
cluster_df

In [ ]:
sample_A = '5dpi_T7'
sample_B = '5dpi_HY'

count_result = pd.read_csv(f'outputs/DEG_edgeR_results/featurecounts.rename.noheader.matrix.{sample_A}_vs_{sample_B}.edgeR.count_matrix', sep = '\t')
de_result = pd.read_csv(f'outputs/DEG_edgeR_results/featurecounts.rename.noheader.matrix.{sample_A}_vs_{sample_B}.edgeR.DE_results', sep = '\t')

de_result = de_result[abs(de_result['logFC']) > 0]

de_result = de_result[de_result['PValue'] <= 0.05]

'''
for i in de_result.index:
    symbol = symbol_df.loc[symbol_df['ENSG']==i, 'Symbol'].item()
    if symbol == '-':
        de_result.loc[i, 'symbol'] = '-'
    else:
        de_result.loc[i, 'symbol'] = symbol'''

for i in de_result.index:
    sampleA_count = count_result.loc[i, f'X{sample_A}']
    sampleB_count = count_result.loc[i, f'X{sample_B}']
    de_result.loc[i, 'sampleA'] = sampleA_count
    de_result.loc[i, 'sampleB'] = sampleB_count
    
de_result = de_result.rename(columns = {'sampleA':sample_A, 'sampleB':sample_B})

de_result = de_result.sort_values(by = 'logFC', ascending = False)
de_result

In [12]:
featureCounts_txt = 'outputs/featureCounts_outputs/3dpi_0_C57mouse_Lung.featurecounts.noheader.txt'
featureCounts_df = pd.read_csv(featureCounts_txt, sep = '\t')
featureCounts_df

,Geneid,Chr,Start,End,Strand,Length,outputs/hisat2_outputs/3dpi_0_C57mouse_Lung.sorted.bam
0,ENSMUSG00000102693,1,3073253,3074322,+,1070,0
1,ENSMUSG00000064842,1,3102016,3102125,+,110,0
2,ENSMUSG00000051951,1;1;1;1;1;1;1,3205901;3206523;3213439;3213609;3214482;342170...,3207317;3207317;3215632;3216344;3216968;342190...,-;-;-;-;-;-;-,6094,2
3,ENSMUSG00000102851,1,3252757,3253236,+,480,0
4,ENSMUSG00000103377,1,3365731,3368549,-,2819,0
...,...,...,...,...,...,...,...
47724,ENSMUSG00000094431,GL456385.1,32719,32818,+,100,0
47725,ENSMUSG00000094621,GL456372.1,13262,13382,-,121,10
47726,ENSMUSG00000098647,GL456381.1,16623,16721,-,99,0
47727,ENSMUSG00000096730,JH584292.1;JH584292.1;JH584292.1;JH584292.1;JH...,3536;3536;3536;3539;3573;5658;5658;5658;5658;6...,3826;3826;3826;3826;3826;5940;5940;5940;5940;7...,+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+,3077,0


In [7]:
for vs in ['HY']:
    for day_post_infection in [3, 5]:
        for sample in ['N289D', 'KT', 'T7']:
            de_result_path = f'vs_{vs}_bak/DEG_edgeR_results_{day_post_infection}dpi/featurecounts.rename.noheader.{day_post_infection}dpi.matrix.{day_post_infection}dpi_{sample}_vs_{day_post_infection}dpi_HY.edgeR.DE_results'
            sample_name = f'{day_post_infection}dpi_{sample}'
            print(sample_name, os.path.exists(de_result_path))



3dpi_N289D True
3dpi_KT True
3dpi_T7 True
5dpi_N289D True
5dpi_KT True
5dpi_T7 True
